# Harmenberg Permanent-Income-Neutral Measure:
# Four-Way Comparison with Sensitivity Analysis

This notebook combines and extends two analyses:

1. **The DemARK [Harmenberg-Aggregation](https://econ-ark.org/materials/harmenberg-aggregation) notebook**: mathematical exposition and MC variance reduction.
2. **The HAFiscal four-way comparison**: validation across Standard MC, Harmenberg MC, Standard 2D TM, and Harmenberg 1D TM.

The key addition is **sensitivity analysis**: how MC errors shrink with sample size $N$, and how TM errors shrink with grid resolution.

## 1. Mathematical Framework

> **Theory reference**: The full derivations of the Harmenberg neutral measure, the aggregation identity, the covariance kernel, and the joint-distribution boundary are in the *BufferStockTheory* appendix `ApndxHarKmenberg` (Carroll 2022). This notebook focuses on **computational** validation and sensitivity analysis.

### Summary of Key Results

For buffer-stock savers with normalized market resources $m_t = M_t / p_t$ and permanent income $p_t$, the consumption function is homothetic: $C(M, p) = c(m) \cdot p$. Harmenberg (2021) defines a neutral measure $\tilde{P}$ by reweighting permanent shock probabilities: $\tilde{\pi}(\psi_j) = \psi_j \cdot \pi(\psi_j)$. The **key identity** is:

$$\mathbb{E}_P[p \cdot c(m)] = \mathbb{E}_{\tilde{P}}[c(m)] \cdot \mathbb{E}_P[p]$$

This eliminates the permanent income dimension from the computational problem for all $p$-linear aggregates (consumption, income, assets, multipliers). For nonlinear-in-$p$ statistics (variance, Gini, welfare), the full joint distribution is still required — see `ApndxHarKmenberg`, Section "When the Joint Distribution Is Required".

### Critical Implementation Detail

The consumption function $c(m)$ must be solved under the **original** measure $P$. The neutral measure $\tilde{P}$ is used only for **simulation/aggregation**, not for solving the Bellman equation.

## 2. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import time
from copy import deepcopy

from HARK.ConsumptionSaving.ConsIndShockModel import IndShockConsumerType
from HARK.ConsumptionSaving.ConsNewKeynesianModel import NewKeynesianConsumerType
from HARK.dual_measure import DualMeasureMixin


class DualIndShock(DualMeasureMixin, IndShockConsumerType):
    """IndShockConsumerType with simultaneous P/Q (Harmenberg) simulation."""
    pass

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

cell_times = {}

### Model Parameters

We use the permanent-income calibration from Carroll, Slacalek, Tokuoka, and White (2017, "cstwMPC"): `LivPrb = 1 − 1/160` (40-year average lifespan) and `PermShkStd = √(0.04/11) ≈ 0.060`, which generates a cross-sectional distribution of permanent income that matches SCF data. The discount factor is set low (`DiscFac = 0.90`) so the buffer-stock ergodic distribution is tightly concentrated and convergence is fast.

In [ ]:
_t0 = time.time()

base_params = {
    "CRRA": 2,
    "Rfree": [1.03],
    "DiscFac": 0.90,
    "LivPrb": [1.0 - 1.0/160.0],          # cstwMPC: 40-year average lifespan
    "PermGroFac": [1.0],
    "PermShkStd": [(0.01*4/11)**0.5],      # cstwMPC: quarterly σ_ψ ≈ 0.060
    "PermShkCount": 7,
    "TranShkStd": [(0.01*4)**0.5],         # cstwMPC: quarterly σ_θ ≈ 0.200
    "TranShkCount": 7,
    "UnempPrb": 0.07,                      # cstwMPC
    "IncUnemp": 0.15,                      # cstwMPC
    "BorrowingConstraint": 0.0,
    "aXtraMax": 50,
    "aXtraCount": 48,
    "T_cycle": 1,
    "cycles": 0,
}

LivPrb = base_params["LivPrb"][0]
PermGroFac = base_params["PermGroFac"][0]
sigma_psi_val = base_params["PermShkStd"][0]
E_p_analytical = (1.0 - LivPrb) / (1.0 - LivPrb * PermGroFac)
sigma_star = np.sqrt(-np.log(LivPrb) - 2*np.log(PermGroFac))
g2_val = PermGroFac**2 * np.exp(sigma_psi_val**2)
print(f"DiscFac             = {base_params['DiscFac']}")
print(f"LivPrb              = {LivPrb:.5f}")
print(f"σ_ψ                 = {sigma_psi_val:.4f}")
print(f"σ_ψ*  (divergence)  = {sigma_star:.4f}  (σ_ψ/σ_ψ* = {sigma_psi_val/sigma_star:.1%})")
print(f"LivPrb × g₂         = {LivPrb * g2_val:.6f}  (must be < 1 for finite Var(p))")
print(f"Analytical E[p]     = {E_p_analytical:.4f}")

cell_times['params'] = time.time() - _t0
print(f"\n[Cell time: {cell_times['params']:.3f}s]")

### Solve the Model

In [ ]:
_t0 = time.time()

agent = IndShockConsumerType(**base_params)
agent.solve()
cFunc = agent.solution[0].cFunc

m_plot = np.linspace(0.01, 5, 200)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(m_plot, cFunc(m_plot), 'b-', linewidth=2)
ax.plot(m_plot, m_plot, 'k--', alpha=0.3, label='45-degree line')
ax.set_xlabel('Normalized market resources $m$')
ax.set_ylabel('Normalized consumption $c(m)$')
ax.set_title('Consumption Function (impatient agent, $\\beta = 0.90$)')
ax.legend()
plt.tight_layout()
plt.show()

cell_times['solve'] = time.time() - _t0
print(f"[Cell time: {cell_times['solve']:.3f}s]")

## 3. The Four Methods

In [ ]:
def run_standard_mc(params, N_agents, T_sim, burn_in=50, seed=0):
    """Method A: Standard MC. Track (m,p), aggregate mean(c*p)."""
    mc = IndShockConsumerType(**params)
    mc.seed = seed
    mc.solve()
    mc.T_sim = T_sim + burn_in
    mc.AgentCount = N_agents
    mc.track_vars = ['cNrm', 'pLvl']
    mc.initialize_sim()
    mc.simulate()
    cNrm = mc.history['cNrm'][burn_in:]
    pLvl = mc.history['pLvl'][burn_in:]
    C_per_period = np.mean(cNrm * pLvl, axis=1)
    return {'C_mean': np.mean(C_per_period), 'C_var': np.var(C_per_period),
            'C_series': C_per_period}


def run_harmenberg_mc(params, N_agents, T_sim, burn_in=50, seed=0):
    """Method B: Harmenberg MC. Solve first, then switch to neutral measure."""
    hm = IndShockConsumerType(**params)
    hm.seed = seed
    hm.solve()
    hm.neutral_measure = True
    hm.construct('IncShkDstn', 'TranShkDstn', 'PermShkDstn')
    hm.T_sim = T_sim + burn_in
    hm.AgentCount = N_agents
    hm.track_vars = ['cNrm']
    hm.initialize_sim()
    hm.simulate()
    cNrm = hm.history['cNrm'][burn_in:]
    LivPrb = params['LivPrb'][0]
    PermGroFac = params['PermGroFac'][0]
    E_p = (1.0 - LivPrb) / (1.0 - LivPrb * PermGroFac)
    C_per_period = np.mean(cNrm, axis=1) * E_p
    return {'C_mean': np.mean(C_per_period), 'C_var': np.var(C_per_period),
            'C_series': C_per_period, 'E_p': E_p}


def run_standard_tm(params, num_m=100, num_p=15):
    """Method C: Standard 2D TM over (m, p) grid."""
    tm = NewKeynesianConsumerType(**params)
    tm.solve()
    tm.define_distribution_grid(num_pointsM=num_m, num_pointsP=num_p)
    tm.calc_transition_matrix()
    tm.calc_ergodic_dist()
    C_agg = float(np.dot(tm.cPol_Grid, np.dot(tm.erg_dstn, tm.dist_pGrid)))
    return {'C_agg': C_agg, 'n_m': len(tm.dist_mGrid),
            'n_p': len(tm.dist_pGrid),
            'n_states': len(tm.dist_mGrid) * len(tm.dist_pGrid)}


def run_harmenberg_tm(params, num_m=100):
    """Method D: Harmenberg 1D TM. Solve first, then switch to neutral measure."""
    hm = NewKeynesianConsumerType(**params)
    hm.solve()
    hm.neutral_measure = True
    hm.construct('IncShkDstn', 'TranShkDstn', 'PermShkDstn')
    hm.define_distribution_grid(num_pointsM=num_m)
    hm.calc_transition_matrix()
    hm.calc_ergodic_dist()
    erg = hm.vec_erg_dstn.flatten()
    C_agg = float(np.dot(hm.cPol_Grid, erg))
    return {'C_agg': C_agg, 'n_m': len(hm.dist_mGrid),
            'n_states': len(hm.dist_mGrid),
            'erg_dstn': erg, 'cPol': hm.cPol_Grid, 'dist_mGrid': hm.dist_mGrid}


def run_dual_mc(params, N_agents, T_sim, burn_in=50, seed=0):
    """Method E: Dual P+Q MC.  Both measures from a single simulation pass.

    Uses DualMeasureMixin to run the standard (P) and Harmenberg (Q)
    simulations simultaneously with shared underlying random draws.
    Returns both P-aggregate and Q-aggregate consumption series.
    """
    dm = DualIndShock(**params)
    dm.seed = seed
    dm.solve()
    dm.setup_Q_measure()
    dm.T_sim = T_sim + burn_in
    dm.AgentCount = N_agents
    dm.track_vars = ['cNrm', 'pLvl']
    dm.initialize_sim()
    dm.simulate()

    cNrm_P = dm.history['cNrm'][burn_in:]
    pLvl_P = dm.history['pLvl'][burn_in:]
    cNrm_Q = dm.history_Q['cNrm'][burn_in:]

    C_P = np.mean(cNrm_P * pLvl_P, axis=1)

    LivPrb = params['LivPrb'][0]
    PermGroFac = params['PermGroFac'][0]
    E_p = (1.0 - LivPrb) / (1.0 - LivPrb * PermGroFac)
    C_Q = np.mean(cNrm_Q, axis=1) * E_p

    return {
        'C_P_mean': np.mean(C_P), 'C_P_var': np.var(C_P), 'C_P_series': C_P,
        'C_Q_mean': np.mean(C_Q), 'C_Q_var': np.var(C_Q), 'C_Q_series': C_Q,
        'E_p': E_p,
    }

## 4. Baseline Cross-Method Comparison

In [ ]:
_t0 = time.time()

print("Method A: Standard MC ...")
t0 = time.time()
res_A = run_standard_mc(base_params, 5000, 200)
time_A = time.time() - t0

print("Method B: Harmenberg MC ...")
t0 = time.time()
res_B = run_harmenberg_mc(base_params, 5000, 200)
time_B = time.time() - t0

print("Method C: Standard 2D TM ...")
t0 = time.time()
res_C = run_standard_tm(base_params, num_m=60, num_p=11)
time_C = time.time() - t0

print("Method D: Harmenberg 1D TM ...")
t0 = time.time()
res_D = run_harmenberg_tm(base_params, num_m=100)
time_D = time.time() - t0

print("Method E: Dual P+Q MC ...")
t0 = time.time()
res_E = run_dual_mc(base_params, 5000, 200)
time_E = time.time() - t0

C_ref = res_D['C_agg']
print(f"\n{'Method':<25} {'C_agg':>10} {'vs D (%)':>10} {'Time (s)':>10}")
print("-" * 58)
for label, C, t in [
    ('A: Standard MC', res_A['C_mean'], time_A),
    ('B: Harmenberg MC', res_B['C_mean'], time_B),
    (f'C: 2D TM ({res_C["n_states"]} st)', res_C['C_agg'], time_C),
    (f'D: 1D TM ({res_D["n_states"]} st)', res_D['C_agg'], time_D),
    ('E: Dual MC (P-track)', res_E['C_P_mean'], time_E),
    ('E: Dual MC (Q-track)', res_E['C_Q_mean'], time_E),
]:
    pct = 100 * (C - C_ref) / C_ref
    print(f"{label:<25} {C:>10.6f} {pct:>+9.4f}% {t:>10.2f}")

print(f"\nVariance reduction (A: Std MC / B: Hrm MC): {res_A['C_var']/max(res_B['C_var'],1e-30):.0f}x")
print(f"Variance reduction (E-P / E-Q):             {res_E['C_P_var']/max(res_E['C_Q_var'],1e-30):.0f}x")
print(f"\nHarmenberg identity check (E_P[c·p] / (E_Q[c]·E_P[p])):")
print(f"  A vs B:  {res_A['C_mean'] / max(res_B['C_mean'], 1e-30):.6f}")
print(f"  E-P vs E-Q: {res_E['C_P_mean'] / max(res_E['C_Q_mean'], 1e-30):.6f}")

cell_times['baseline'] = time.time() - _t0
print(f"\n[Cell time: {cell_times['baseline']:.3f}s]")

## 5. MC Sensitivity: Variance vs Sample Size

In [ ]:
_t0 = time.time()

agent_sizes = [100, 300, 1000, 3000, 10000]
T_var = 200

var_C_std = []
var_C_hrm = []

for N in agent_sizes:
    print(f"N = {N:>6d} ... ", end='', flush=True)
    r_s = run_standard_mc(base_params, N, T_var)
    r_h = run_harmenberg_mc(base_params, N, T_var)
    var_C_std.append(r_s['C_var'])
    var_C_hrm.append(r_h['C_var'])
    ratio = var_C_std[-1] / max(var_C_hrm[-1], 1e-30)
    print(f"Var ratio = {ratio:>8.1f}x")

var_C_std = np.array(var_C_std)
var_C_hrm = np.array(var_C_hrm)

cell_times['mc_var_sweep'] = time.time() - _t0
print(f"\n[Cell time: {cell_times['mc_var_sweep']:.3f}s]")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(agent_sizes, var_C_std, 'rs-', label='Standard MC', linewidth=2, markersize=8)
ax.loglog(agent_sizes, var_C_hrm, 'bo-', label='Harmenberg MC', linewidth=2, markersize=8)
ax.set_xlabel('Number of agents $N$')
ax.set_ylabel('Variance of per-period aggregate $\\hat{C}_t$')
ax.set_title('MC Variance Reduction: Standard vs Harmenberg')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print(f"Variance ratios (Std / Hrm):")
for N, r in zip(agent_sizes, var_C_std / np.maximum(var_C_hrm, 1e-30)):
    print(f"  N = {N:>6d}: {r:>8.0f}x")

### MC Point Estimate Convergence

In [ ]:
_t0 = time.time()

mc_sizes = [300, 1000, 3000, 10000]
n_seeds = 3
C_ref_tm = res_D['C_agg']
std_means, std_errs, hrm_means, hrm_errs = [], [], [], []

for N in mc_sizes:
    print(f"N = {N:>6d} ... ", end='', flush=True)
    cs, ch = [], []
    for s in range(n_seeds):
        cs.append(run_standard_mc(base_params, N, 150, seed=s)['C_mean'])
        ch.append(run_harmenberg_mc(base_params, N, 150, seed=s)['C_mean'])
    std_means.append(np.mean(cs)); std_errs.append(np.std(cs))
    hrm_means.append(np.mean(ch)); hrm_errs.append(np.std(ch))
    print(f"Std: {std_means[-1]:.5f} +/- {std_errs[-1]:.5f}, "
          f"Hrm: {hrm_means[-1]:.5f} +/- {hrm_errs[-1]:.5f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.axhline(C_ref_tm, color='green', linestyle='--', linewidth=2,
           label=f'1D TM reference = {C_ref_tm:.5f}')
ax.errorbar(mc_sizes, std_means, yerr=std_errs,
            fmt='rs-', capsize=4, linewidth=1.5, label='Standard MC')
ax.errorbar(mc_sizes, hrm_means, yerr=hrm_errs,
            fmt='bo-', capsize=4, linewidth=1.5, label='Harmenberg MC')
ax.set_xscale('log')
ax.set_xlabel('Number of agents $N$')
ax.set_ylabel('Aggregate consumption $\\hat{C}$')
ax.set_title('MC Point Estimate Convergence')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

cell_times['mc_convergence'] = time.time() - _t0
print(f"[Cell time: {cell_times['mc_convergence']:.3f}s]")

## 6. TM Sensitivity: Error vs Grid Resolution

### 6a. Sensitivity to $n_m$

In [ ]:
_t0 = time.time()

m_counts = [15, 30, 50, 75, 100, 150]

C_1d, C_2d, time_1d, time_2d = [], [], [], []

for nm in m_counts:
    print(f"n_m = {nm:>4d} ... ", end='', flush=True)
    t0 = time.time()
    r1 = run_harmenberg_tm(base_params, num_m=nm)
    t1 = time.time() - t0
    t0 = time.time()
    r2 = run_standard_tm(base_params, num_m=nm, num_p=11)
    t2 = time.time() - t0
    C_1d.append(r1['C_agg']); time_1d.append(t1)
    C_2d.append(r2['C_agg']); time_2d.append(t2)
    print(f"1D: {r1['C_agg']:.6f} ({t1:.2f}s), 2D: {r2['C_agg']:.6f} ({t2:.2f}s)")

C_ref_fine = C_1d[-1]
err_1d = np.abs(np.array(C_1d) - C_ref_fine) / abs(C_ref_fine) * 100
err_2d = np.abs(np.array(C_2d) - C_ref_fine) / abs(C_ref_fine) * 100

cell_times['tm_nm_sweep'] = time.time() - _t0
print(f"\n[Cell time: {cell_times['tm_nm_sweep']:.3f}s]")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.semilogy(m_counts[:-1], err_1d[:-1], 'bo-', label='Harmenberg 1D TM', linewidth=2, markersize=8)
ax1.semilogy(m_counts, np.maximum(err_2d, 1e-4), 'rs-', label='Standard 2D TM', linewidth=2, markersize=8)
ax1.set_xlabel('Number of $m$ grid points $n_m$')
ax1.set_ylabel('Relative error in $C$ (%, vs finest 1D)')
ax1.set_title('Accuracy vs $n_m$')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.semilogy(m_counts, time_1d, 'bo-', label='Harmenberg 1D TM', linewidth=2, markersize=8)
ax2.semilogy(m_counts, time_2d, 'rs-', label='Standard 2D TM', linewidth=2, markersize=8)
ax2.set_xlabel('Number of $m$ grid points $n_m$')
ax2.set_ylabel('Time (seconds)')
ax2.set_title('Computation Time vs $n_m$')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6b. Sensitivity to $n_p$ (2D TM Only)

In [ ]:
_t0 = time.time()

p_counts_requested = [7, 11, 15, 21, 25]
nm_fixed = 60
C_2d_p, time_2d_p, actual_np, n_states_p = [], [], [], []

for np_req in p_counts_requested:
    t0 = time.time()
    r = run_standard_tm(base_params, num_m=nm_fixed, num_p=np_req)
    t = time.time() - t0
    C_2d_p.append(r['C_agg']); time_2d_p.append(t)
    actual_np.append(r['n_p']); n_states_p.append(r['n_states'])
    print(f"n_p req={np_req:>3d}, actual={r['n_p']:>3d}, "
          f"states={r['n_states']:>6d}, C={r['C_agg']:.6f}, {t:.2f}s")

r1d_ref = run_harmenberg_tm(base_params, num_m=nm_fixed)
C_1d_ref = r1d_ref['C_agg']
err_p = np.abs(np.array(C_2d_p) - C_1d_ref) / abs(C_1d_ref) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.semilogy(actual_np, np.maximum(err_p, 1e-4), 'rs-', linewidth=2, markersize=8)
ax1.set_xlabel('Actual $n_p$ grid points')
ax1.set_ylabel('Relative error in $C$ (%, vs 1D TM)')
ax1.set_title('2D TM: Sensitivity to $p$ Grid')
ax1.grid(True, alpha=0.3)
ax2.plot(n_states_p, time_2d_p, 'rs-', linewidth=2, markersize=8)
ax2.set_xlabel('Total states ($n_m \\times n_p$)')
ax2.set_ylabel('Time (seconds)')
ax2.set_title('2D TM: Time vs State Space Size')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n1D TM reference: C = {C_1d_ref:.6f}  (n_m={nm_fixed})")
print(f"{'n_p':>6} {'states':>8} {'C_agg':>10} {'Err%':>8} {'Time':>7}")
for np_, ns, C, e, t in zip(actual_np, n_states_p, C_2d_p, err_p, time_2d_p):
    print(f"{np_:>6d} {ns:>8d} {C:>10.6f} {e:>7.4f}% {t:>6.2f}s")

cell_times['tm_np_sweep'] = time.time() - _t0
print(f"\n[Cell time: {cell_times['tm_np_sweep']:.3f}s]")

## 7. Ergodic Distribution Comparison

In [ ]:
_t0 = time.time()

mc_erg = IndShockConsumerType(**base_params)
mc_erg.solve()
mc_erg.T_sim = 200; mc_erg.AgentCount = 10000
mc_erg.track_vars = ['mNrm']
mc_erg.initialize_sim(); mc_erg.simulate()
m_std_sample = mc_erg.history['mNrm'][-1, :]

hm_erg = IndShockConsumerType(**base_params)
hm_erg.solve()
hm_erg.neutral_measure = True
hm_erg.construct('IncShkDstn', 'TranShkDstn', 'PermShkDstn')
hm_erg.T_sim = 200; hm_erg.AgentCount = 10000
hm_erg.track_vars = ['mNrm']
hm_erg.initialize_sim(); hm_erg.simulate()
m_hrm_sample = hm_erg.history['mNrm'][-1, :]

r1d_fine = run_harmenberg_tm(base_params, num_m=150)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(m_std_sample, bins=80, range=(0, 5), density=True, alpha=0.35,
        color='red', label='Standard MC (10k, measure $P$)')
ax.hist(m_hrm_sample, bins=80, range=(0, 5), density=True, alpha=0.35,
        color='blue', label='Harmenberg MC (10k, measure $\\tilde{P}$)')

dm = r1d_fine['dist_mGrid']
erg = r1d_fine['erg_dstn']
bw = np.diff(np.concatenate([[0], (dm[:-1] + dm[1:]) / 2, [dm[-1] * 1.1]]))
mask = dm < 5
ax.plot(dm[mask], (erg / bw)[mask], 'g-', linewidth=2,
        label='1D TM ergodic (150 pts, $\\tilde{P}$)')

ax.set_xlabel('Normalized market resources $m$')
ax.set_ylabel('Density')
ax.set_title('Ergodic Distribution of $m$ ($\\beta = 0.90$, tightly concentrated)')
ax.set_xlim(0, 5)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

cell_times['ergodic_dist'] = time.time() - _t0
print(f"[Cell time: {cell_times['ergodic_dist']:.3f}s]")

## 8. Covariance Kernel and 1D Approximation Experiments

> **Theory reference**: BST `ApndxHarKmenberg`, Sections "The Covariance Kernel and 1D Computability" and "Higher-Order Moments from Analytical $p$-Structure".

The Harmenberg neutral measure makes aggregate *levels* (consumption, income, assets) cheap to compute from the 1D distribution. But *inequality* statistics (variance, Gini, quantile shares) are nonlinear in $p$ and formally require the full joint distribution.

This section tests two approximation strategies that avoid the expensive 2D TM:

1. **Covariance kernel** $\gamma(a)$: computes $\text{Cov}(c_{\text{nrm}}, p)$ *exactly* from 1D objects.
2. **Analytical $p$-moments + independence**: approximates $\text{Var}(A)$ by $\mathbb{E}[p^2]\,\mathbb{E}[a^2] - (\mathbb{E}[p]\,\mathbb{E}[a])^2$, exploiting asymptotic independence of $p$ and $m$.

We compare both against ground truth from the Standard 2D TM and Standard MC.

In [ ]:
_t0 = time.time()

# ── Model parameters ──
Rfree  = base_params["Rfree"][0]
LivPrb = base_params["LivPrb"][0]
G      = base_params["PermGroFac"][0]   # PermGroFac
CRRA   = base_params["CRRA"]
sigma_psi = base_params["PermShkStd"][0]

# ── Analytical moments of p ──
E_p  = (1.0 - LivPrb) / (1.0 - LivPrb * G)   # mean permanent income
g2   = G**2 * np.exp(sigma_psi**2)              # per-period growth factor for E[p²]
E_p2 = (1.0 - LivPrb) / (1.0 - LivPrb * g2)   # E[p²] (perpetual-youth steady state)
Var_p = E_p2 - E_p**2

print("=== Analytical permanent-income moments ===")
print(f"  E[p]   = {E_p:.6f}")
print(f"  E[p²]  = {E_p2:.6f}")
print(f"  Var(p) = {Var_p:.6f}")
print(f"  CV(p)  = {np.sqrt(Var_p)/E_p:.4f}")

cell_times['cov_setup'] = time.time() - _t0
print(f"\n[Cell time: {cell_times['cov_setup']:.3f}s]")

### 8a. Ground Truth from 2D TM and Standard MC

In [ ]:
_t0 = time.time()

# ── MC ground truth (primary reference) ──
# MC naturally handles the p-distribution regardless of its variance.
# We compute CROSS-SECTIONAL statistics per period, then average.
mc_ref = IndShockConsumerType(**base_params)
mc_ref.solve()
cFunc = mc_ref.solution[0].cFunc
mc_ref.T_sim = 500
mc_ref.AgentCount = 10000
mc_ref.track_vars = ['cNrm', 'pLvl', 'mNrm', 'aNrm']
mc_ref.initialize_sim()
mc_ref.simulate()

burn = 200
T_use = mc_ref.T_sim - burn
cNrm_hist = mc_ref.history['cNrm'][burn:]    # (T_use, N)
pLvl_hist = mc_ref.history['pLvl'][burn:]
aNrm_hist = mc_ref.history['aNrm'][burn:]

# Cross-sectional covariance per period, then average
Cov_cp_per_t = np.array([np.cov(cNrm_hist[t], pLvl_hist[t])[0, 1] for t in range(T_use)])
Cov_cp_mc = np.mean(Cov_cp_per_t)

# Cross-sectional Var(A = p*a) per period
VarA_per_t = np.array([np.var(pLvl_hist[t] * aNrm_hist[t]) for t in range(T_use)])
Var_A_mc = np.mean(VarA_per_t)

# Cross-sectional Var(C_lvl = p*c) per period
VarC_per_t = np.array([np.var(pLvl_hist[t] * cNrm_hist[t]) for t in range(T_use)])
Var_C_mc = np.mean(VarC_per_t)

C_mc = np.mean(cNrm_hist * pLvl_hist)

print(f"=== MC Ground Truth (N={mc_ref.AgentCount}, T={T_use} periods) ===")
print(f"  C (agg)       = {C_mc:.6f}")
print(f"  Cov(c,p)      = {Cov_cp_mc:.8f}  (± {np.std(Cov_cp_per_t)/np.sqrt(T_use):.8f})")
print(f"  Cov/C ratio   = {Cov_cp_mc/C_mc*100:.4f}%")
print(f"  Var(A=p*a)    = {Var_A_mc:.6f}  (± {np.std(VarA_per_t)/np.sqrt(T_use):.6f})")
print(f"  Var(C=p*c)    = {Var_C_mc:.6f}")
print(f"\nNote: with Var(p) = {Var_p:.1f} (CV = {np.sqrt(Var_p)/E_p:.1f}), the 2D TM")
print(f"cannot adequately represent the p-distribution with typical grid sizes.")
print(f"MC is the primary ground truth for joint-distribution statistics.")

cell_times['cov_truth'] = time.time() - _t0
print(f"\n[Cell time: {cell_times['cov_truth']:.3f}s]")

### 8b. Covariance Kernel $\gamma(a)$ from 1D Objects

The covariance kernel (BST `ApndxHarKmenberg` eq. \eqref{eq:CovKernel}):

$$\gamma(a) = \text{Cov}_{\psi,\theta}\!\left(\psi,\; c\!\left(\frac{R\,a}{\Gamma\,\psi} + \theta\right)\right)$$

is computed for each grid point $a_k$ by summing over the discrete shock distribution. Then:

$$\text{Cov}(c_{\text{nrm}}, p) = \bar{p} \cdot \Gamma \cdot \sum_k \tilde{\chi}^m[k] \cdot \gamma(a(m_k))$$

where $\tilde{\chi}^m$ is the 1D Harmenberg (permanent-income-weighted) ergodic distribution.

In [ ]:
_t0 = time.time()

num_m_1d = 100

# ── Build 1D Harmenberg TM (Q-distribution) ──
hm1d = NewKeynesianConsumerType(**base_params)
hm1d.solve()
hm1d.neutral_measure = True
hm1d.construct('IncShkDstn', 'TranShkDstn', 'PermShkDstn')
hm1d.define_distribution_grid(num_pointsM=num_m_1d)
hm1d.calc_transition_matrix()
hm1d.calc_ergodic_dist()

m_grid_1d  = hm1d.dist_mGrid
c_grid_1d  = hm1d.cPol_Grid
a_grid_1d  = hm1d.aPol_Grid
erg_Q      = hm1d.vec_erg_dstn.flatten()

print(f"Harmenberg 1D C = {float(np.dot(c_grid_1d, erg_Q)) * E_p:.6f}  (vs MC {C_mc:.6f})")

# ── Extract shock distribution (ORIGINAL measure P) ──
agent_P = IndShockConsumerType(**base_params)
agent_P.solve()
shk_dstn_P = agent_P.IncShkDstn[0]
probs_P   = shk_dstn_P.pmv
perm_shks = shk_dstn_P.atoms[0]
tran_shks = shk_dstn_P.atoms[1]

print(f"\nShock distribution: {len(probs_P)} points")
print(f"  E_P[ψ]  = {np.dot(probs_P, perm_shks):.6f} (should be ≈1)")
print(f"  E_P[ψ²] = {np.dot(probs_P, perm_shks**2):.6f} (analytical: {np.exp(sigma_psi**2):.6f})")

# ── Compute covariance kernel γ(a_k) ──
def covariance_kernel(a_vals, cFunc, Rfree, PermGroFac, probs, perm_shks, tran_shks):
    """Compute γ(a) = Cov_ψ,θ(ψ, c(R*a/(G*ψ) + θ)) for each a."""
    gamma = np.zeros(len(a_vals))
    for k, a_k in enumerate(a_vals):
        m_next = Rfree * a_k / (PermGroFac * perm_shks) + tran_shks
        c_next = np.array([float(cFunc(m)) for m in m_next])
        E_psi_c  = np.dot(probs, perm_shks * c_next)
        E_psi    = np.dot(probs, perm_shks)
        E_c      = np.dot(probs, c_next)
        gamma[k] = E_psi_c - E_psi * E_c
    return gamma

gamma_vals = covariance_kernel(a_grid_1d, cFunc, Rfree, G, probs_P, perm_shks, tran_shks)

# ── Compute Cov(c, p) from Q-weighted kernel ──
Cov_cp_Q = E_p * G * float(np.dot(erg_Q, gamma_vals))

# ── Also compute from MC agent sample to validate the kernel itself ──
a_sample = aNrm_hist[-1, :500]
gamma_mc_sample = covariance_kernel(a_sample, cFunc, Rfree, G, probs_P, perm_shks, tran_shks)
Cov_cp_mc_kernel = E_p * G * float(np.mean(gamma_mc_sample))

print(f"\n=== Covariance Kernel Results ===")
print(f"  γ(a) range: [{gamma_vals.min():.8f}, {gamma_vals.max():.8f}]")
print(f"")
print(f"  Cov(c,p) from Q-weighted kernel  = {Cov_cp_Q:.8f}")
print(f"  Cov(c,p) from MC-sample kernel   = {Cov_cp_mc_kernel:.8f}")
print(f"  Cov(c,p) from MC cross-section   = {Cov_cp_mc:.8f}")
print(f"")
if abs(Cov_cp_mc) > 1e-12:
    print(f"  Q-kernel vs MC rel error     = {abs(Cov_cp_Q - Cov_cp_mc)/abs(Cov_cp_mc)*100:.1f}%")
    print(f"  MC-kernel vs MC cross-sect   = {abs(Cov_cp_mc_kernel - Cov_cp_mc)/abs(Cov_cp_mc)*100:.1f}%")
print(f"  Sign: {'negative (ψ-channel)' if Cov_cp_Q < 0 else 'positive'}")
print(f"\nNote: the 1D kernel formula and the direct MC cross-sectional")
print(f"covariance measure the same quantity. The kernel approach avoids")
print(f"tracking p-levels entirely — it needs only the m-distribution and shocks.")

cell_times['cov_kernel'] = time.time() - _t0
print(f"\n[Cell time: {cell_times['cov_kernel']:.3f}s]")

In [ ]:
# ── Plot the covariance kernel ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: γ(a) as a function of a
ax = axes[0]
ax.plot(a_grid_1d, gamma_vals, 'b-', linewidth=2)
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel('End-of-period assets $a$')
ax.set_ylabel('$\\gamma(a)$')
ax.set_title('Covariance Kernel $\\gamma(a) = \\mathrm{Cov}_{\\psi,\\theta}(\\psi, c(Ra/(\\Gamma\\psi)+\\theta))$')
ax.set_xlim(0, min(5, a_grid_1d.max()))

# Right: weighted kernel γ(a) * π_Q(m_k)
ax = axes[1]
weighted_gamma = gamma_vals * erg_Q
ax.bar(a_grid_1d, weighted_gamma, width=np.diff(np.append(a_grid_1d, a_grid_1d[-1]*1.1))*0.8,
       color='steelblue', alpha=0.7, label='$\\tilde{\\chi}^m[k] \\cdot \\gamma(a_k)$')
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel('End-of-period assets $a$')
ax.set_ylabel('Weighted contribution')
ax.set_title('Contribution to $\\mathrm{Cov}(c,p)$ by asset level')
ax.set_xlim(0, min(5, a_grid_1d.max()))
ax.legend()

plt.tight_layout()
plt.show()

### 8c. Higher-Order Moment Approximation: $\text{Var}(A)$

The variance of level assets $A = p \cdot a(m)$ decomposes as:

$$\text{Var}(A) = \mathbb{E}[p^2 a^2] - (\mathbb{E}[pa])^2$$

The **independence approximation** (BST `ApndxHarKmenberg`, Section "Higher-Order Moments") uses asymptotic independence of $p$ and $m$ in the ergodic distribution:

$$\text{Var}(A) \approx \mathbb{E}[p^2]\,\mathbb{E}[a^2] - (\mathbb{E}[p])^2\,(\mathbb{E}[a])^2$$

where $\mathbb{E}[p^2]$ comes from the analytical Blanchard–Yaari formula and $\mathbb{E}[a^2]$ from the 1D Harmenberg distribution.

In [ ]:
_t0 = time.time()

# ── 1D moments from Harmenberg Q-distribution ──
E_a_Q  = float(np.dot(a_grid_1d, erg_Q))
E_a2_Q = float(np.dot(a_grid_1d**2, erg_Q))
E_c_Q  = float(np.dot(c_grid_1d, erg_Q))
E_c2_Q = float(np.dot(c_grid_1d**2, erg_Q))
Var_a_Q = E_a2_Q - E_a_Q**2
Var_c_Q = E_c2_Q - E_c_Q**2

# Under asymptotic independence of p and m:
# Var(p*a) = E[p²]·E[a²] - (E[p]·E[a])²
#          = Var(p)·E[a]² + Var(a)·E[p]² + Var(p)·Var(a)
Var_A_indep = E_p2 * E_a2_Q - (E_p * E_a_Q)**2
Var_C_indep = E_p2 * E_c2_Q - (E_p * E_c_Q)**2

# MC cross-sectional truth (computed in previous cell)
# Var_A_mc, Var_C_mc

print("=== Higher-Order Moment Approximation: Independence Method ===\n")
print(f"{'Statistic':<25} {'MC Truth':>14} {'1D+Analytical':>14} {'Rel Error':>10}")
print("-" * 68)
if Var_A_mc > 0:
    print(f"{'Var(A = p·a)':<25} {Var_A_mc:>14.6f} {Var_A_indep:>14.6f} "
          f"{abs(Var_A_indep - Var_A_mc)/Var_A_mc*100:>9.1f}%")
    print(f"{'SD(A)':<25} {np.sqrt(Var_A_mc):>14.6f} {np.sqrt(max(Var_A_indep,0)):>14.6f}")
if Var_C_mc > 0:
    print(f"{'Var(C = p·c)':<25} {Var_C_mc:>14.6f} {Var_C_indep:>14.6f} "
          f"{abs(Var_C_indep - Var_C_mc)/Var_C_mc*100:>9.1f}%")

print(f"\n--- Component Decomposition (under independence) ---")
print(f"  Var(p)  [analytical] = {Var_p:.4f}   (CV = {np.sqrt(Var_p)/E_p:.2f})")
print(f"  Var(a)  [1D Q-dist]  = {Var_a_Q:.6f}")
print(f"  Var(c)  [1D Q-dist]  = {Var_c_Q:.6f}")
print(f"  E[a]    [1D Q-dist]  = {E_a_Q:.6f}")
print(f"  E[c]    [1D Q-dist]  = {E_c_Q:.6f}")
print(f"")
print(f"  Var(p)·E[a]²  = {Var_p * E_a_Q**2:.4f}   (dominant when Var(p) is large)")
print(f"  Var(a)·E[p]²  = {Var_a_Q * E_p**2:.6f}")
print(f"  Var(p)·Var(a) = {Var_p * Var_a_Q:.4f}")
print(f"  Sum            = {Var_p*E_a_Q**2 + Var_a_Q*E_p**2 + Var_p*Var_a_Q:.4f} (≈ Var_A_indep)")

# Compare MC component estimates
E_a_mc = np.mean(aNrm_hist[burn:].flatten())
Var_a_mc_direct = np.mean([np.var(aNrm_hist[t]) for t in range(T_use)])
E_p_mc = np.mean(pLvl_hist.flatten())
Var_p_mc = np.mean([np.var(pLvl_hist[t]) for t in range(T_use)])

print(f"\n--- MC component estimates (for comparison) ---")
print(f"  E[a]  MC = {E_a_mc:.6f}   vs 1D = {E_a_Q:.6f}")
print(f"  Var(a) MC = {Var_a_mc_direct:.6f}   vs 1D = {Var_a_Q:.6f}")
print(f"  E[p]  MC = {E_p_mc:.6f}   vs analytical = {E_p:.6f}")
print(f"  Var(p) MC = {Var_p_mc:.4f}   vs analytical = {Var_p:.4f}")

cell_times['cov_higherorder'] = time.time() - _t0
print(f"\n[Cell time: {cell_times['cov_higherorder']:.3f}s]")

### 8d. Sensitivity: How Approximation Quality Varies with Parameters

The independence approximation works because $p$ and $m$ are asymptotically independent in the ergodic distribution. But the quality depends on:
- **Shock variance** ($\sigma_\psi$): larger shocks → more $p$-dispersion → independence more important
- **Grid resolution** ($n_m$): finer grid → better 1D moments
- **Survival probability** ($L$): longer lives → more $p$-dispersion

We sweep $\sigma_\psi$ to test how the approximation degrades.

In [ ]:
_t0 = time.time()

sigma_star_sweep = np.sqrt(-np.log(LivPrb) - 2*np.log(G))
sigma_vals = [0.02, 0.04, 0.06, 0.065, 0.070, 0.075]
print(f"σ_ψ* = {sigma_star_sweep:.4f}  (Var(p) diverges above this)\n")
results_sweep = []

for sig in sigma_vals:
    params_i = {**base_params, "PermShkStd": [sig]}

    # Analytical p-moments
    g2_i   = G**2 * np.exp(sig**2)
    wp_g2  = LivPrb * g2_i
    if wp_g2 >= 1.0:
        print(f"σ_ψ = {sig:.3f}: DIVERGENT (ℓ·g₂ = {wp_g2:.4f} ≥ 1) — skipping")
        continue
    E_p2_i = (1.0 - LivPrb) / (1.0 - wp_g2)
    Var_p_i = E_p2_i - E_p**2

    # 1D Harmenberg TM
    hm_i = NewKeynesianConsumerType(**params_i)
    hm_i.solve()
    cFunc_i = hm_i.solution[0].cFunc
    hm_i.neutral_measure = True
    hm_i.construct('IncShkDstn', 'TranShkDstn', 'PermShkDstn')
    hm_i.define_distribution_grid(num_pointsM=80)
    hm_i.calc_transition_matrix()
    hm_i.calc_ergodic_dist()
    erg_i = hm_i.vec_erg_dstn.flatten()
    a_i   = hm_i.aPol_Grid

    E_a2_i = float(np.dot(a_i**2, erg_i))
    E_a_i  = float(np.dot(a_i, erg_i))
    Var_A_approx_i = E_p2_i * E_a2_i - (E_p * E_a_i)**2

    # Covariance kernel
    agent_Pi = IndShockConsumerType(**params_i)
    agent_Pi.solve()
    shk_i = agent_Pi.IncShkDstn[0]
    gamma_i = covariance_kernel(a_i, cFunc_i, Rfree, G, shk_i.pmv, shk_i.atoms[0], shk_i.atoms[1])
    Cov_cp_i = E_p * G * float(np.dot(erg_i, gamma_i))

    # MC truth (smaller sample for speed in sweep)
    mc_i = IndShockConsumerType(**params_i)
    mc_i.solve()
    mc_i.T_sim = 300; mc_i.AgentCount = 5000
    mc_i.track_vars = ['cNrm', 'pLvl', 'aNrm']
    mc_i.initialize_sim(); mc_i.simulate()
    burn_i = 100; T_i = mc_i.T_sim - burn_i
    cN_i = mc_i.history['cNrm'][burn_i:]
    pL_i = mc_i.history['pLvl'][burn_i:]
    aN_i = mc_i.history['aNrm'][burn_i:]
    Cov_mc_i = np.mean([np.cov(cN_i[t], pL_i[t])[0,1] for t in range(T_i)])
    VarA_mc_i = np.mean([np.var(pL_i[t] * aN_i[t]) for t in range(T_i)])

    results_sweep.append({
        'sigma': sig, 'Var_p': Var_p_i,
        'Cov_1d': Cov_cp_i, 'Cov_mc': Cov_mc_i,
        'VarA_approx': Var_A_approx_i, 'VarA_mc': VarA_mc_i,
    })
    cov_err_i = abs(Cov_cp_i - Cov_mc_i) / max(abs(Cov_mc_i), 1e-12) * 100
    varA_err_i = abs(Var_A_approx_i - VarA_mc_i) / max(abs(VarA_mc_i), 1e-12) * 100
    print(f"σ_ψ = {sig:.2f}: Var(p) = {Var_p_i:>8.1f}, "
          f"Cov 1D={Cov_cp_i:>10.6f} MC={Cov_mc_i:>10.6f} err={cov_err_i:>5.1f}%, "
          f"VarA 1D={Var_A_approx_i:>8.3f} MC={VarA_mc_i:>8.3f} err={varA_err_i:>5.1f}%")

cell_times['cov_sweep'] = time.time() - _t0
print(f"\n[Cell time: {cell_times['cov_sweep']:.3f}s]")

In [ ]:
sigmas     = [r['sigma'] for r in results_sweep]
cov_err    = [abs(r['Cov_1d'] - r['Cov_mc']) / max(abs(r['Cov_mc']), 1e-12) * 100 for r in results_sweep]
varA_err   = [abs(r['VarA_approx'] - r['VarA_mc']) / max(abs(r['VarA_mc']), 1e-12) * 100 for r in results_sweep]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: Cov(c,p) — 1D kernel vs MC truth
ax = axes[0]
ax.plot(sigmas, [r['Cov_1d'] for r in results_sweep], 'bo-', label='1D kernel', linewidth=2, markersize=8)
ax.plot(sigmas, [r['Cov_mc'] for r in results_sweep], 'rs--', label='MC truth', linewidth=2, markersize=8)
ax.set_xlabel('$\\sigma_\\psi$ (perm. shock std)')
ax.set_ylabel('$\\mathrm{Cov}(c_{\\mathrm{nrm}}, p)$')
ax.set_title('Cov from 1D Kernel vs MC')
ax.legend()
ax.grid(True, alpha=0.3)

# Middle: Relative errors
ax = axes[1]
ax.semilogy(sigmas, np.maximum(cov_err, 0.1), 'bo-', label='Cov(c,p) — 1D kernel', linewidth=2, markersize=8)
ax.semilogy(sigmas, np.maximum(varA_err, 0.1), 'rs-', label='Var(A) — independence', linewidth=2, markersize=8)
ax.set_xlabel('$\\sigma_\\psi$')
ax.set_ylabel('Relative error vs MC (%)')
ax.set_title('Approximation Error vs Shock Size')
ax.legend()
ax.grid(True, alpha=0.3, which='both')

# Right: Var(A) — approx vs MC truth
ax = axes[2]
ax.plot(sigmas, [r['VarA_approx'] for r in results_sweep], 'bo-', label='1D+analytical approx', linewidth=2, markersize=8)
ax.plot(sigmas, [r['VarA_mc'] for r in results_sweep], 'rs--', label='MC truth', linewidth=2, markersize=8)
ax.set_xlabel('$\\sigma_\\psi$')
ax.set_ylabel('$\\mathrm{Var}(A)$')
ax.set_title('Var(A) from 1D+Analytical vs MC')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 8e. Grid Sensitivity for the Covariance Kernel

The covariance kernel $\gamma(a)$ computation uses the 1D Harmenberg grid. How sensitive is the result to the number of $m$-grid points?

In [ ]:
_t0 = time.time()

nm_sweep = [15, 30, 50, 75, 100, 150]
Cov_by_nm = []
VarA_by_nm = []

for nm in nm_sweep:
    hm_s = NewKeynesianConsumerType(**base_params)
    hm_s.solve()
    cFunc_s = hm_s.solution[0].cFunc
    hm_s.neutral_measure = True
    hm_s.construct('IncShkDstn', 'TranShkDstn', 'PermShkDstn')
    hm_s.define_distribution_grid(num_pointsM=nm)
    hm_s.calc_transition_matrix()
    hm_s.calc_ergodic_dist()
    erg_s = hm_s.vec_erg_dstn.flatten()
    a_s   = hm_s.aPol_Grid

    gamma_s = covariance_kernel(a_s, cFunc_s, Rfree, G, probs_P, perm_shks, tran_shks)
    Cov_s = E_p * G * np.dot(erg_s, gamma_s)
    Cov_by_nm.append(Cov_s)

    E_a2_s = float(np.dot(a_s**2, erg_s))
    E_a_s  = float(np.dot(a_s, erg_s))
    VarA_s = E_p2 * E_a2_s - (E_p * E_a_s)**2
    VarA_by_nm.append(VarA_s)

    print(f"n_m = {nm:>4d}: Cov(c,p) = {Cov_s:.8f}, Var(A) = {VarA_s:.6f}")

# Use finest grid as reference
Cov_ref = Cov_by_nm[-1]
VarA_ref = VarA_by_nm[-1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.semilogy(nm_sweep[:-1],
             [abs(c - Cov_ref)/max(abs(Cov_ref), 1e-12)*100 for c in Cov_by_nm[:-1]],
             'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Number of $m$-grid points')
ax1.set_ylabel('Relative error vs finest (%)')
ax1.set_title('Cov(c,p) from 1D Kernel: Grid Convergence')
ax1.grid(True, alpha=0.3, which='both')

ax2.semilogy(nm_sweep[:-1],
             [abs(v - VarA_ref)/max(abs(VarA_ref), 1e-12)*100 for v in VarA_by_nm[:-1]],
             'rs-', linewidth=2, markersize=8)
ax2.set_xlabel('Number of $m$-grid points')
ax2.set_ylabel('Relative error vs finest (%)')
ax2.set_title('Var(A) from 1D+Analytical: Grid Convergence')
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

cell_times['cov_grid_sweep'] = time.time() - _t0
print(f"\n[Cell time: {cell_times['cov_grid_sweep']:.3f}s]")

### 8g. Lorenz Curve Reconstruction from 1D Objects

The Lorenz curve of level wealth $A_i = p_i \cdot a(m_i)$ requires the full joint distribution (see §8f). Can we approximate it from 1D Harmenberg objects plus analytical $p$-moments?

We test four reconstruction strategies:

1. **Naive 1D**: Lorenz curve of $a(m)$ alone, ignoring $p$-dispersion entirely.
2. **Independence**: Draw $p$ from the analytical age-conditional lognormal and $a$ from the 1D Q-distribution, paired independently. Uses $\mathbb{E}[p^k]$ but assumes $p \perp a$.
3. **Kernel-corrected**: Use a Gaussian copula with rank correlation derived from the 1D covariance kernel's $\text{Cov}(a, p)$ (Term A only).
4. **Oracle-corrected**: Gaussian copula with rank correlation from the MC sample (showing the ceiling of what any correction could achieve).

All are compared against the **MC ground truth**.

In [ ]:
_t0 = time.time()

from scipy.stats import norm as sp_norm, rankdata

def lorenz_curve(values, weights=None):
    """Compute Lorenz curve (cum_pop, cum_wealth) from values and optional weights."""
    values = np.asarray(values, dtype=float)
    if weights is None:
        weights = np.ones(len(values))
    weights = np.asarray(weights, dtype=float)
    mask = weights > 0
    values, weights = values[mask], weights[mask]
    idx = np.argsort(values)
    values, weights = values[idx], weights[idx]
    cum_w = np.cumsum(weights)
    cum_w = cum_w / cum_w[-1]
    cum_val = np.cumsum(values * weights)
    cum_val = cum_val / cum_val[-1]
    cum_w = np.concatenate([[0], cum_w])
    cum_val = np.concatenate([[0], cum_val])
    return cum_w, cum_val

def gini_from_lorenz(cum_pop, cum_val):
    return 1 - 2 * np.trapz(cum_val, cum_pop)

def sample_p_analytical(n_samples, LivPrb, PermGroFac, sigma_psi, rng):
    """Draw p from the ergodic cross-sectional distribution (perpetual youth)."""
    ages = rng.geometric(1 - LivPrb, size=n_samples)
    log_p = np.zeros(n_samples)
    for i, age in enumerate(ages):
        if age > 0:
            log_p[i] = rng.normal(-age * sigma_psi**2 / 2, sigma_psi * np.sqrt(age))
    return np.exp(log_p)

def sample_a_from_Q(n_samples, a_grid, erg_Q, rng):
    """Draw a-values from the 1D Harmenberg distribution."""
    erg_Q_pos = np.maximum(erg_Q, 0)
    erg_Q_pos /= erg_Q_pos.sum()
    idx = rng.choice(len(a_grid), size=n_samples, p=erg_Q_pos)
    return a_grid[idx]

def gaussian_copula_pair(n_samples, rho, rng):
    """Draw (u, v) pairs from a Gaussian copula with correlation rho."""
    z1 = rng.normal(size=n_samples)
    z2 = rho * z1 + np.sqrt(1 - rho**2) * rng.normal(size=n_samples)
    u = sp_norm.cdf(z1)
    v = sp_norm.cdf(z2)
    return u, v

def quantile_function(cdf_vals, grid, weights):
    """Given uniform samples u, map to values via the weighted empirical CDF."""
    w = np.maximum(weights, 0)
    w /= w.sum()
    idx = np.argsort(grid)
    sorted_grid = grid[idx]
    sorted_w = w[idx]
    cum_w = np.cumsum(sorted_w)
    cum_w[-1] = 1.0
    return np.interp(cdf_vals, cum_w, sorted_grid)

def quantile_p_analytical(u_vals, LivPrb, PermGroFac, sigma_psi, n_age=2000, rng=None):
    """Map uniform quantiles to p-values via the analytical CDF (mixture of lognormals)."""
    if rng is None:
        rng = np.random.default_rng(42)
    big_sample = sample_p_analytical(200_000, LivPrb, PermGroFac, sigma_psi, rng)
    big_sample.sort()
    n = len(big_sample)
    empirical_cdf = np.arange(1, n + 1) / n
    return np.interp(u_vals, empirical_cdf, big_sample)

# ── Parameters ──
n_synth = 200_000
LivPrb_val = base_params['LivPrb'][0]
rng = np.random.default_rng(2026)

# ── 1. MC ground truth Lorenz (use last 50 periods, all agents) ──
T_lorenz = 50
A_mc_pool = (pLvl_hist[-T_lorenz:] * aNrm_hist[-T_lorenz:]).flatten()
A_mc_pool = A_mc_pool[A_mc_pool > 0]
lz_mc_pop, lz_mc_val = lorenz_curve(A_mc_pool)
gini_mc = gini_from_lorenz(lz_mc_pop, lz_mc_val)

# ── 2. Naive 1D: Lorenz of a(m) from Q-distribution ──
lz_1d_pop, lz_1d_val = lorenz_curve(a_grid_1d, erg_Q)
gini_1d = gini_from_lorenz(lz_1d_pop, lz_1d_val)

# ── 3. Independence reconstruction ──
a_indep = sample_a_from_Q(n_synth, a_grid_1d, erg_Q, rng)
p_indep = sample_p_analytical(n_synth, LivPrb_val, G, sigma_psi, rng)
A_indep = p_indep * a_indep
A_indep = A_indep[A_indep > 0]
lz_indep_pop, lz_indep_val = lorenz_curve(A_indep)
gini_indep = gini_from_lorenz(lz_indep_pop, lz_indep_val)

# ── 4. Kernel-corrected: Gaussian copula with Cov from 1D kernel ──
# Cov(a, p) ≈ E[p] * G * E_Q[γ_a(a)] where γ_a uses the asset analog
# For rank correlation, we use Cov/(SD_a * SD_p)
Cov_ap_kernel = Cov_cp_Q  # Cov(c,p) as proxy for the sign/direction
SD_a_Q = np.sqrt(Var_a_Q)
SD_p = np.sqrt(Var_p) if Var_p > 0 else 1.0
rho_kernel = Cov_ap_kernel / (SD_a_Q * SD_p) if SD_a_Q > 0 else 0.0
rho_kernel = np.clip(rho_kernel, -0.99, 0.99)

u_k, v_k = gaussian_copula_pair(n_synth, rho_kernel, rng)
a_copula_k = quantile_function(u_k, a_grid_1d, erg_Q)
p_copula_k = quantile_p_analytical(v_k, LivPrb_val, G, sigma_psi, rng=np.random.default_rng(99))
A_copula_k = p_copula_k * a_copula_k
A_copula_k = A_copula_k[A_copula_k > 0]
lz_kern_pop, lz_kern_val = lorenz_curve(A_copula_k)
gini_kern = gini_from_lorenz(lz_kern_pop, lz_kern_val)

# ── 5. Oracle-corrected: use MC rank correlation ──
a_mc_flat = aNrm_hist[-T_lorenz:].flatten()
p_mc_flat = pLvl_hist[-T_lorenz:].flatten()
rho_mc = np.corrcoef(a_mc_flat, p_mc_flat)[0, 1]

u_o, v_o = gaussian_copula_pair(n_synth, rho_mc, rng)
a_copula_o = quantile_function(u_o, a_grid_1d, erg_Q)
p_copula_o = quantile_p_analytical(v_o, LivPrb_val, G, sigma_psi, rng=np.random.default_rng(99))
A_copula_o = p_copula_o * a_copula_o
A_copula_o = A_copula_o[A_copula_o > 0]
lz_oracle_pop, lz_oracle_val = lorenz_curve(A_copula_o)
gini_oracle = gini_from_lorenz(lz_oracle_pop, lz_oracle_val)

print("=== Lorenz / Gini Reconstruction ===\n")
print(f"{'Method':<25} {'Gini':>8} {'Error vs MC':>14}")
print("-" * 50)
print(f"{'MC truth':<25} {gini_mc:>8.4f} {'—':>14}")
print(f"{'Naive 1D (a only)':<25} {gini_1d:>8.4f} {gini_1d - gini_mc:>+14.4f}")
print(f"{'Independence (p⊥a)':<25} {gini_indep:>8.4f} {gini_indep - gini_mc:>+14.4f}")
print(f"{'Kernel-corrected':<25} {gini_kern:>8.4f} {gini_kern - gini_mc:>+14.4f}")
print(f"{'Oracle-corrected':<25} {gini_oracle:>8.4f} {gini_oracle - gini_mc:>+14.4f}")
print(f"\nRank correlations:")
print(f"  ρ(a,p) from 1D kernel = {rho_kernel:.4f}")
print(f"  ρ(a,p) from MC sample = {rho_mc:.4f}")

cell_times['lorenz_reconstruct'] = time.time() - _t0
print(f"\n[Cell time: {cell_times['lorenz_reconstruct']:.3f}s]")

In [ ]:
_t0 = time.time()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Left panel: Lorenz curves ──
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect equality')
ax.plot(lz_mc_pop, lz_mc_val, 'b-', linewidth=2.5, label=f'MC truth (Gini={gini_mc:.3f})')
ax.plot(lz_1d_pop, lz_1d_val, 'g:', linewidth=2, label=f'Naive 1D (Gini={gini_1d:.3f})')
ax.plot(lz_indep_pop, lz_indep_val, 'r--', linewidth=1.8, label=f'Indep p⊥a (Gini={gini_indep:.3f})')
ax.plot(lz_kern_pop, lz_kern_val, 'm-.', linewidth=1.8, label=f'Kernel-corr (Gini={gini_kern:.3f})')
ax.plot(lz_oracle_pop, lz_oracle_val, 'c-', linewidth=1.5, alpha=0.8,
        label=f'Oracle-corr (Gini={gini_oracle:.3f})')
ax.set_xlabel('Cumulative population share')
ax.set_ylabel('Cumulative wealth share')
ax.set_title('Lorenz Curves of Level Wealth A = p · a')
ax.legend(fontsize=8, loc='upper left')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

# ── Right panel: difference from MC truth (zoomed) ──
ax2 = axes[1]
n_pts = 500
pop_grid = np.linspace(0, 1, n_pts)
mc_interp = np.interp(pop_grid, lz_mc_pop, lz_mc_val)
for pop, val, label, color, ls in [
    (lz_1d_pop, lz_1d_val, 'Naive 1D', 'green', ':'),
    (lz_indep_pop, lz_indep_val, 'Independence', 'red', '--'),
    (lz_kern_pop, lz_kern_val, 'Kernel-corr', 'purple', '-.'),
    (lz_oracle_pop, lz_oracle_val, 'Oracle-corr', 'cyan', '-'),
]:
    interp_val = np.interp(pop_grid, pop, val)
    ax2.plot(pop_grid, interp_val - mc_interp, color=color, linestyle=ls, linewidth=1.8, label=label)
ax2.axhline(0, color='blue', linewidth=0.8, alpha=0.5)
ax2.set_xlabel('Cumulative population share')
ax2.set_ylabel('Lorenz deviation from MC truth')
ax2.set_title('Reconstruction Error')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('Figures/lorenz_reconstruction.png', dpi=150, bbox_inches='tight')
plt.show()

cell_times['lorenz_plot'] = time.time() - _t0
print(f"[Cell time: {cell_times['lorenz_plot']:.3f}s]")

### Wealth Shares and Copula Sensitivity

In [ ]:
_t0 = time.time()

def wealth_shares(values, weights=None, quantiles=[0.50, 0.90, 0.99]):
    """Compute share of total wealth held above each quantile."""
    values = np.asarray(values, dtype=float)
    if weights is None:
        weights = np.ones(len(values))
    weights = np.asarray(weights, dtype=float)
    mask = weights > 0
    values, weights = values[mask], weights[mask]
    idx = np.argsort(values)
    values, weights = values[idx], weights[idx]
    cum_w = np.cumsum(weights)
    cum_w /= cum_w[-1]
    total_val = np.sum(values * weights)
    shares = {}
    for q in quantiles:
        i_q = np.searchsorted(cum_w, q)
        top_share = np.sum(values[i_q:] * weights[i_q:]) / total_val
        shares[q] = top_share
    return shares

# ── Percentile shares for each method ──
print("=== Wealth Concentration: Share of Total Wealth ===\n")
print(f"{'Method':<25} {'Bottom 50%':>11} {'Top 10%':>11} {'Top 1%':>11} {'Gini':>8}")
print("-" * 70)

methods = [
    ("MC truth", A_mc_pool, None),
]
for name, vals, w in methods:
    sh = wealth_shares(vals, w)
    g = gini_mc
    print(f"{name:<25} {1-sh[0.50]:>11.3%} {sh[0.90]:>11.3%} {sh[0.99]:>11.3%} {g:>8.4f}")

for name, pop, val, g in [
    ("Naive 1D", lz_1d_pop, lz_1d_val, gini_1d),
    ("Independence", lz_indep_pop, lz_indep_val, gini_indep),
    ("Kernel-corrected", lz_kern_pop, lz_kern_val, gini_kern),
    ("Oracle-corrected", lz_oracle_pop, lz_oracle_val, gini_oracle),
]:
    bot50 = np.interp(0.50, pop, val)
    top10 = 1 - np.interp(0.90, pop, val)
    top1 = 1 - np.interp(0.99, pop, val)
    print(f"{name:<25} {bot50:>11.3%} {top10:>11.3%} {top1:>11.3%} {g:>8.4f}")

# ── Sweep: Gini and top-10% share vs assumed rank correlation ──
rho_grid = np.linspace(-0.3, 0.5, 17)
gini_sweep = []
top10_sweep = []
rng_sweep = np.random.default_rng(777)

for rho_try in rho_grid:
    rho_clipped = np.clip(rho_try, -0.99, 0.99)
    u_s, v_s = gaussian_copula_pair(n_synth, rho_clipped, rng_sweep)
    a_s = quantile_function(u_s, a_grid_1d, erg_Q)
    p_s = quantile_p_analytical(v_s, LivPrb_val, G, sigma_psi,
                                rng=np.random.default_rng(99))
    A_s = p_s * a_s
    A_s = A_s[A_s > 0]
    lz_p, lz_v = lorenz_curve(A_s)
    gini_sweep.append(gini_from_lorenz(lz_p, lz_v))
    top10_sweep.append(1 - np.interp(0.90, lz_p, lz_v))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(rho_grid, gini_sweep, 'ko-', markersize=4)
ax1.axhline(gini_mc, color='blue', linewidth=1.5, label=f'MC truth = {gini_mc:.4f}')
ax1.axvline(rho_kernel, color='purple', linestyle='-.', alpha=0.7, label=f'Kernel ρ = {rho_kernel:.3f}')
ax1.axvline(rho_mc, color='cyan', linestyle='--', alpha=0.7, label=f'MC ρ = {rho_mc:.3f}')
ax1.set_xlabel('Assumed rank correlation ρ(a, p)')
ax1.set_ylabel('Gini coefficient')
ax1.set_title('Gini vs Copula Correlation')
ax1.legend(fontsize=8)

ax2.plot(rho_grid, top10_sweep, 'ko-', markersize=4)
sh_mc = wealth_shares(A_mc_pool)
ax2.axhline(sh_mc[0.90], color='blue', linewidth=1.5, label=f'MC truth = {sh_mc[0.90]:.3f}')
ax2.axvline(rho_kernel, color='purple', linestyle='-.', alpha=0.7, label=f'Kernel ρ')
ax2.axvline(rho_mc, color='cyan', linestyle='--', alpha=0.7, label=f'MC ρ')
ax2.set_xlabel('Assumed rank correlation ρ(a, p)')
ax2.set_ylabel('Top 10% wealth share')
ax2.set_title('Top 10% Share vs Copula Correlation')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('Figures/lorenz_copula_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

cell_times['lorenz_shares'] = time.time() - _t0
print(f"\n[Cell time: {cell_times['lorenz_shares']:.3f}s]")

### 8h. Summary of Lorenz Curve Experiments

The Lorenz curve of level wealth $A = p \cdot a$ is a nonlinear-in-$p$ statistic that cannot be computed from the 1D Harmenberg distribution alone.  The experiments above test whether combining the 1D distribution with analytical $p$-moments and a copula correction can approximate the true Lorenz curve.

**Key findings:**

| Approach | What it uses | Expected accuracy |
|----------|-------------|-------------------|
| Naive 1D ($a$ only) | $\pi_Q(m)$ | Severely underestimates inequality (ignores all $p$-dispersion) |
| Independence ($p \perp a$) | $\pi_Q(m)$ + analytical $p$-marginal | Captures $p$-dispersion but misses $(p, a)$ correlation |
| Kernel-corrected | Above + $\text{Cov}(a,p)$ from 1D kernel | Adjusts for within-shock correlation (Term A); misses accumulated between-agent correlation (Term B) |
| Oracle-corrected | Above + true $\rho(a,p)$ from MC | Shows ceiling of copula approach; residual error comes from non-Gaussian dependence structure |

The copula sensitivity sweep reveals how the Gini and top-10% wealth share vary continuously with the assumed rank correlation $\rho(a,p)$, enabling calibration of the copula parameter from even limited MC information.

### 8j. Aggregate MPC and $p$-Weighting (Covariance Structure)

Define the **$p$-weighted average marginal propensity to consume** (normalized MPC, $\partial c_{\mathrm{nrm}}/\partial m$):

$$\chi := \mathbb{E}\bigl[p \cdot c'(m)\bigr]$$

where $c'(m) = \partial c/\partial m$ is the derivative of the *normalized* consumption function. This object shows up when aggregating responses to shocks that scale with permanent income or when comparing heterogeneous agents in efficiency units: it is **linear in $p$ for fixed $m$**, but because $c'(m)$ varies with $m$ and $m$ is jointly distributed with $p$, the aggregate is **not** $\mathbb{E}[p]\,\mathbb{E}[c'(m)]$ unless $\mathrm{Cov}(p, c'(m))=0$.

By the usual decomposition,

$$\chi = \mathbb{E}[p]\,\mathbb{E}[c'(m)] + \mathrm{Cov}\bigl(p, c'(m)\bigr).$$

The Harmenberg neutral measure implies the **exact 1D identity** (BST `ApndxHarKmenberg`, same logic as for $\mathbb{E}[p\,c(m)]$)

$$\mathbb{E}_P\bigl[p \cdot g(m)\bigr] = \mathbb{E}_P[p]\cdot \mathbb{E}_Q\bigl[g(m)\bigr]$$

for any measurable $g$ of normalized resources $m$ — in particular $g(m)=c'(m)$. So the **Harmenberg 1D TM** delivers $\chi$ from $\mathbb{E}_P[p]$ and the $Q$-ergodic weights on $m$, with **no separate covariance correction**. The exercises below parallel §8g: **MC truth**; a **naive product** $\bar p \times \widehat{\mathbb{E}}[c']$ that drops $\mathrm{Cov}(p,c')$; **Harmenberg 1D TM** ($\mathbb{E}_p[p]\,\mathbb{E}_Q[c']$); an **uncorrected 1D TM** that uses the physical $m$-marginal with $p$ collapsed to a single grid point and then multiplies by $\mathbb{E}[p]$ (wrong weights); **standard 2D TM** on $(m,p)$; and **copula** reconstructions (independence vs.\ oracle rank correlation for $(p,m)$).

In [ ]:
_t0 = time.time()

def mpc_nrm_from_cFunc(cFunc, m_vals):
    """∂c_nrm/∂m at each grid point or sample (HARK LowerEnvelope supports .derivative)."""
    m_vals = np.asarray(m_vals, dtype=float)
    return np.array([float(cFunc.derivative(float(x))) for x in m_vals.ravel()]).reshape(m_vals.shape)


def sample_m_from_Q(n_samples, m_grid, erg_Q, rng):
    w = np.maximum(np.asarray(erg_Q).flatten(), 0.0)
    w /= w.sum()
    idx = rng.choice(len(m_grid), size=n_samples, p=w)
    return m_grid[idx]

# ── MC truth (reuse burn-in simulation from §8a) ──
m_flat = mNrm_hist[burn:].flatten()
p_flat = pLvl_hist[burn:].flatten()
mpc_flat = mpc_nrm_from_cFunc(cFunc, m_flat)
chi_mc = float(np.mean(p_flat * mpc_flat))
E_mpc_mc = float(np.mean(mpc_flat))
Cov_p_mpc = float(np.cov(p_flat, mpc_flat, ddof=1)[0, 1])
chi_naive_product = float(E_p * E_mpc_mc)
chi_decomp_check = float(E_p * E_mpc_mc + Cov_p_mpc)

# ── Harmenberg 1D TM (exact χ = E_p · E_Q[c'] under theory) ──
mpc_hm = mpc_nrm_from_cFunc(cFunc, m_grid_1d)
chi_hberg_1d = float(E_p * np.dot(mpc_hm, erg_Q))

# ── "Uncorrected" 1D: standard (physical) TM on m only with p collapsed to 1, then × E_p
#     (wrong weights — not the Harmenberg Q marginal)
tm1d_phys = NewKeynesianConsumerType(**base_params)
tm1d_phys.solve()
tm1d_phys.neutral_measure = False
tm1d_phys.define_distribution_grid(num_pointsM=num_m_1d, dist_pGrid=np.array([1.0]))
tm1d_phys.calc_transition_matrix()
tm1d_phys.calc_ergodic_dist()
erg_phys = tm1d_phys.vec_erg_dstn.flatten()
mpc_phys = mpc_nrm_from_cFunc(cFunc, tm1d_phys.dist_mGrid)
chi_uncorrected_1d = float(E_p * np.dot(mpc_phys, erg_phys))

# ── Standard 2D TM (joint π on (m, p)) ──
num_m_2d, num_p_2d = 50, 9
tm2d = NewKeynesianConsumerType(**base_params)
tm2d.solve()
tm2d.neutral_measure = False
tm2d.define_distribution_grid(num_pointsM=num_m_2d, num_pointsP=num_p_2d)
tm2d.calc_transition_matrix()
tm2d.calc_ergodic_dist()
pi2d = tm2d.erg_dstn
m2d, p2d = tm2d.dist_mGrid, tm2d.dist_pGrid
mpc_m2d = mpc_nrm_from_cFunc(cFunc, m2d)
chi_2d = float(np.sum(pi2d * (p2d[np.newaxis, :] * mpc_m2d[:, np.newaxis])))

# ── Copula reconstructions on (p, m): same Gaussian copula as §8g ──
n_synth_mpc = 120_000
rng_mpc = np.random.default_rng(31415)
LivPrb_mpc = base_params["LivPrb"][0]

m_ind = sample_m_from_Q(n_synth_mpc, m_grid_1d, erg_Q, rng_mpc)
p_ind = sample_p_analytical(n_synth_mpc, LivPrb_mpc, G, sigma_psi, rng_mpc)
chi_copula_indep = float(np.mean(p_ind * mpc_nrm_from_cFunc(cFunc, m_ind)))

rho_pm = np.corrcoef(m_flat, p_flat)[0, 1]
rho_use = float(np.clip(rho_pm, -0.99, 0.99))
u_o, v_o = gaussian_copula_pair(n_synth_mpc, rho_use, rng_mpc)
m_orc = quantile_function(u_o, m_grid_1d, erg_Q)
p_orc = quantile_p_analytical(v_o, LivPrb_mpc, G, sigma_psi, rng=np.random.default_rng(99))
chi_copula_oracle = float(np.mean(p_orc * mpc_nrm_from_cFunc(cFunc, m_orc)))

print("=== Aggregate χ = E[ p · ∂c/∂m ] (normalized MPC) ===\n")
print(f"{'Method':<32} {'χ':>12} {'vs MC (%)':>12}")
print("-" * 58)
rows = [
    ("MC truth", chi_mc),
    ("Naive: Ē[p]·E[c′] (drops Cov)", chi_naive_product),
    ("Decomp check: Ē[p]E[c′]+Cov̂", chi_decomp_check),
    ("Harmenberg 1D (E_p·E_Q[c′])", chi_hberg_1d),
    ("Uncorrected 1D TM × E_p (p≡1 slice)", chi_uncorrected_1d),
    (f"Standard 2D TM ({num_m_2d}×{len(p2d)} st)", chi_2d),
    ("Copula: independence (p⊥m)", chi_copula_indep),
    (f"Copula: oracle ρ(m,p)={rho_pm:.3f}", chi_copula_oracle),
]
for name, x in rows:
    rel = 100.0 * (x - chi_mc) / max(abs(chi_mc), 1e-12)
    print(f"{name:<32} {x:>12.6f} {rel:>+11.2f}%")
print("-" * 58)
print(f"\nCov(p, c′) [MC sample] = {Cov_p_mpc:.8f}")
print(f"Share of χ from covariance: {Cov_p_mpc/chi_mc*100:.2f}% of χ")
print(f"\nNote: Harmenberg theory predicts χ = E_p·E_Q[c′]; any gap vs MC is discretization / simulation noise.")

cell_times['agg_mpc'] = time.time() - _t0
print(f"\n[Cell time: {cell_times['agg_mpc']:.3f}s]")

### 8i. Summary of Covariance, Approximation, Lorenz, and Aggregate-MPC Experiments

#### What the covariance kernel captures — and what it misses

The law of total covariance decomposes $\text{Cov}(c_{\text{nrm}}, p)$ into:

$$\text{Cov}(c_t, p_t) = \underbrace{\mathbb{E}[\text{Cov}(c_t, p_t \mid a_{t-1}, p_{t-1})]}_{\text{Term A: within-shock}} + \underbrace{\text{Cov}(\mathbb{E}[c_t \mid \cdot], \mathbb{E}[p_t \mid \cdot])}_{\text{Term B: between-agent}}$$

The 1D covariance kernel $\gamma(a)$ computes **Term A exactly**: the covariance arising from the current period's shocks, which creates instantaneous negative correlation between $\psi$ (raising $p$) and $c(m')$ (lowered via the $\psi$-channel in $m' = Ra/(\Gamma\psi) + \theta$).

**Term B** is the cross-sectional covariance between agents' *expected* consumption and their permanent income, accumulated from their entire shock history. The BST appendix derivation assumes Term B $\approx 0$ by asymptotic independence of $p$ and $a$. The experiments show this assumption is **too strong for long-lived agents** (`LivPrb` $= 0.99$): Term B accounts for $\sim$2/3 of the total covariance, because agents accumulate correlated $(p, a)$ states over many periods.

#### When the independence approximation breaks down

| Regime | $\text{LivPrb} \cdot g_2$ | $\mathbb{E}[p^2]$ | Cov kernel accuracy | Var(A) approx |
|--------|--------------------------|-------------------|-------------------|---------------|
| $\sigma_\psi$ small | $< 1$ | Finite, small | ~75% of truth (Term B small) | Good (~10%) |
| $\sigma_\psi$ moderate | $\lesssim 1$ | Finite, large | ~30% of truth | Very poor |
| $\sigma_\psi$ large | $> 1$ | **Diverges** ($\infty$) | Captures only Term A | Meaningless |

Here $g_2 = \Gamma^2 \mathbb{E}[\psi^2] = \Gamma^2 e^{\sigma_\psi^2}$. When $\text{LivPrb} \cdot g_2 > 1$, the second moment of $p$ diverges, and the independence approximation for $\text{Var}(A)$ is undefined.

#### Positive results

- The **covariance kernel itself** is well-defined and computable from 1D objects for any $\sigma_\psi$.
- The kernel's $m$-grid **convergence is rapid**: $n_m = 50$ suffices for sub-1% accuracy.
- The covariance is **consistently negative** (the $\psi$-channel), confirming the theoretical sign prediction.
- The **magnitude** of Term A (within-shock covariance) is reliably captured.

#### Lorenz curve reconstruction

The Lorenz curve experiments (§8g–8h) show that combining the 1D Harmenberg distribution with the analytical $p$-marginal via a Gaussian copula provides a practical path for approximating inequality statistics:

- **Independence** ($p \perp a$): Captures most of the $p$-driven inequality, often the dominant source.
- **Kernel-corrected**: The 1D covariance kernel provides a rank correlation that partially corrects the copula pairing. Because it captures only Term A, the correction undershoots when between-agent correlation is large.
- **Oracle-corrected**: Using the true MC rank correlation in the copula gives Gini and wealth shares close to the MC truth, with residual error from the Gaussian copula's inability to capture tail dependence.
- **Copula sensitivity**: The Gini and wealth shares vary smoothly with $\rho$, so even a rough estimate of the rank correlation (e.g., from a small/short MC run) substantially improves the 1D reconstruction.

#### Aggregate $p$-weighted MPC ($\chi = \mathbb{E}[p\,c'(m)]$, §8j)

The object $\chi$ is still a **joint** statistic because $c'(m)$ varies with $m$, but the integrand is **linear in $p$** for given $m$. Harmenberg's aggregation identity therefore applies: $\chi = \mathbb{E}_P[p]\cdot \mathbb{E}_Q[c'(m)]$, so the **Harmenberg 1D TM** delivers the correct $\chi$ without a separate covariance correction. A **naive** $\bar p \cdot \widehat{\mathbb{E}}[c'(m)]$ that uses the sample mean of $c'(m)$ but ignores $\mathrm{Cov}(p,c'(m))$ generally **misses** $\chi$ whenever that covariance is non-zero. An **uncorrected 1D TM** that fixes $p$ on the grid and scales by $\mathbb{E}[p]$ uses the wrong $m$-marginal and can be far off. **Standard 2D TM** and **copula** reconstructions for $(p,m)$ parallel §8g; in the cstwMPC-style calibration here, $\mathrm{Cov}(p,c')$ is typically small, so naive, Harmenberg 1D, 2D TM, and copula estimates cluster tightly.

#### Implications for HAFiscal

For HAFiscal's calibration (moderate $\sigma_\psi$, education-group heterogeneity), the 1D kernel provides a useful **lower bound** on $|\text{Cov}(c,p)|$. The full covariance is larger due to the between-agent accumulation channel, but for computing multipliers and spending levels — which are $p$-linear and don't need $\text{Cov}(c,p)$ at all — the Harmenberg neutral measure remains fully exact.

For inequality/welfare statistics, the copula reconstruction offers a middle ground: run a short MC simulation to estimate $\rho(a,p)$, then use the 1D TM distribution + copula for the detailed distributional analysis. This is far cheaper than a full 2D TM while being more accurate than the pure independence assumption.

## 9. Cell Timing Summary

In [ ]:
total = sum(cell_times.values())
print(f"{'Cell':<25} {'Time (s)':>10} {'Share':>8}")
print('-' * 45)
for name, t in sorted(cell_times.items(), key=lambda x: -x[1]):
    print(f"{name:<25} {t:>10.2f} {100*t/total:>7.1f}%")
print('-' * 45)
print(f"{'TOTAL':<25} {total:>10.2f}")

## 10. Summary

### Key Findings

1. **Harmenberg MC reduces variance by 2+ orders of magnitude** relative to standard MC at the same $N$.

2. **Harmenberg 1D TM dramatically reduces the state space** from $n_m \times n_p$ to $n_m$.

3. **All four methods converge to the same aggregate** when properly implemented (solve under standard measure, simulate/aggregate under neutral measure).

4. **TM methods converge rapidly with $n_m$**: beyond ~50 grid points, accuracy is typically within 0.1%.

5. **Critical implementation detail**: The consumption function must be solved under the original measure $P$. Solving under the neutral measure $\tilde{P}$ gives a distorted consumption function.

### Theory Reference

The general mathematical results — the change of measure, the aggregation identity, the covariance kernel and its 1D computability, and the characterization of which statistics require the full joint distribution — are derived in the *BufferStockTheory* appendix `ApndxHarKmenberg` (Carroll 2022, Sections "Harmenberg's Method", "When the Joint Distribution Is Required", "The Covariance Kernel and 1D Computability", and "Higher-Order Moments"). This notebook provides the computational validation of those results.

### References

- Harmenberg, Karl (2021). "Aggregation with a permanent income shock." *JEDC*, 129, 104185.
- Carroll, Christopher D. (2022). *Theoretical Foundations of Buffer Stock Saving*. Appendix `ApndxHarKmenberg`.
- Carroll, Christopher D. et al. "The Econ-ARK and HARK." [econ-ark.org](https://econ-ark.org)